In [0]:

from pyspark.sql import functions as F

# Validación del motor Spark
print("Spark:", spark.version)

# Identificación del catálogo activo
spark.sql("SELECT current_catalog(), current_schema()").show()

# Prueba funcional de PySpark
df = spark.range(1, 1001)

df = df.withColumn(
    "valor_calculado",
    F.col("id") * 10
)

print("Registros:", df.count())

display(df.limit(10))


In [0]:

# Configuración inicial de Unity Catalog

CATALOG = "workspace"
PROJECT = "codi"

SCHEMAS = {
    "bronze": f"{CATALOG}.{PROJECT}_bronze",
    "silver": f"{CATALOG}.{PROJECT}_silver",
    "gold": f"{CATALOG}.{PROJECT}_gold"
}

for layer, schema in SCHEMAS.items():
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {schema}"
    )
    print(f"OK - {layer.upper()}: {schema}")

# Verificar esquemas disponibles
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))


In [0]:

from pyspark.sql import functions as F

TABLE = "workspace.codi_bronze.test_ingestion"

df_test = (
    spark.range(1, 1001)
    .withColumn("source_system", F.lit("CODI_TEST"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("record_status", F.lit("VALID"))
)

# Primera persistencia en Delta Lake
(
    df_test.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLE)
)

print(f"Tabla creada: {TABLE}")

# Validación de persistencia
result = spark.table(TABLE)

print(f"Registros persistidos: {result.count()}")

display(result.limit(10))


In [0]:

# Historial transaccional de la tabla Delta
display(
    spark.sql("""
        DESCRIBE HISTORY
        workspace.codi_bronze.test_ingestion
    """)
)
